# IMDb animation series 2022: custom score

Reads `data.csv` (the full scrape, ~1.27 million rows, **not in this repo**). If it is missing, falls back to `data_sample.csv`, a small sample recovered from the `*_x` columns of the committed `out.csv`. **With the sample, the means/maxima (`vm, rm, pm, pM, vM`) differ from the full-data ones, so `cal` values differ from the committed `out.csv`.** Results go to `out.csv` for the full data and `out_sample.csv` for the sample.

In [ ]:
import os
import numpy as np
import pandas as pd
from scoring import clean_numbers, cal_score, join_unambiguous  # the logic lives in scoring.py so it can be tested

In [ ]:
FULL = os.path.exists('data.csv')
alldata = pd.read_csv('data.csv' if FULL else 'data_sample.csv', dtype=str)
series = pd.read_csv('series22.csv', dtype=str)
print('USING FULL data.csv' if FULL else 'USING SMALL data_sample.csv: numbers are NOT comparable to the full-data out.csv')
print(len(alldata), 'rows in alldata;', len(series), 'rows in series22')

Clean: strip thousands separators and trailing dots, convert to numbers (bad values become NaN and are dropped).

In [ ]:
alldata = clean_numbers(alldata)
series = clean_numbers(series)
alldata.dtypes

Statistics: means (`vm, rm, pm`) and maxima (`pM, vM`) of votes, rating and place.

In [ ]:
vm = alldata['vote'].mean()
rm = alldata['rate'].mean()
pm = alldata['place'].mean()
pM = alldata['place'].max()
vM = alldata['vote'].max()
print(len(alldata), vm, rm, pm, pM, vM)

Custom score `cal`: the original eight-term formula (four weighted pairs), vectorized (the original loops/`apply` cells were slow and ran out of memory on the full data). The formula itself is in `scoring.py`.

In [ ]:
alldata['cal'] = cal_score(alldata['vote'], alldata['place'], alldata['rate'], vm, rm, pm, pM, vM)
alldata.head()

Join with `series22.csv` on title (the committed data has no IMDb id; if both files have an `imdb_id` column, that is used instead). Titles that occur more than once on either side are ambiguous (different shows share a name), so they are dropped instead of producing duplicate/mismatched rows.

In [ ]:
key = 'imdb_id' if 'imdb_id' in alldata.columns and 'imdb_id' in series.columns else 'title'
com = join_unambiguous(alldata, series, key)
com.head(10)

In [ ]:
out = 'out.csv' if FULL else 'out_sample.csv'
com.to_csv(out, index=False)
print(len(com), 'rows written to', out)